# Guía: cómo usar el forward de PyLBLRTM desde el repo

Esta guía es para quien **clonó el repositorio** y quiere calcular transmitancias y atenuaciones. La versión de PyPI (`pip install pylblrtm`) es vieja (0.1.1, sin forward): todo lo de aquí funciona **solo instalando desde el repo**.

Cada sección corre en segundos con rangos espectrales pequeños; la primera vez descarga líneas de HITRAN (necesita internet) y quedan en caché.

**Contenido**

1. Instalación desde el repo
2. Las tres piezas del forward
3. Forward rápido en Python: un gas en un camino standoff
4. Varios gases a la vez
5. Todo desde un YAML con `run_path` (por gas, bines en µm, dB/m)
6. Atmósfera vertical (columna, nadir o inclinada)
7. Parámetros del cálculo y cuándo cambiarlos
8. Errores y excepciones
9. Unidades, convenciones y limitaciones

## 1. Instalación desde el repo

En una terminal (no en el notebook):

```bash
git clone https://github.com/OscarMigue9/PyLBLRTM.git
cd PyLBLRTM
git checkout dev
python -m venv .venv
.venv/bin/python -m pip install -e ".[fast,notebook]"      # Windows: .venv\Scripts\python
```

| Instalación | Agrega | Para qué |
|---|---|---|
| `pip install -e .` | numpy, scipy, hitran-api, pyyaml | Lo mínimo: descargar líneas y calcular |
| `.[fast]` | numba | Backend `humlicek`: decenas de veces más rápido |
| `.[notebook]` | matplotlib, ipykernel, nbconvert | Gráficas y notebooks |
| `.[dev]` | pytest | Correr los tests: `python -m pytest tests` |

`-e` (editable) hace que un `git pull` actualice la librería sin reinstalar. Luego abre este notebook con el kernel del `.venv`.

> **Si alguna vez instalaste `pip install pylblrtm`** (la 0.1.1 de PyPI), ese Python sigue usando la versión vieja hasta que corras `pip install -e .` desde el repo en ese mismo Python. La celda siguiente lo detecta.

La celda siguiente verifica la instalación.

In [ ]:
import importlib.util
import pylblrtm

print("pylblrtm", pylblrtm.__version__, "desde", pylblrtm.__file__)
# La versión de PyPI (0.1.1) no tiene el forward: si quedó instalada de antes, Python la usa
# en lugar del código del repo
if not hasattr(pylblrtm, "run_path"):
    raise ImportError(
        "Este Python usa una versión vieja de pylblrtm, no la del repo. Desde la carpeta del "
        "repo corre:  python -m pip install -e \".[fast,notebook]\"  y reinicia el kernel."
    )
for mod, para in [("numba", "voigt='humlicek' (rápido)"), ("matplotlib", "gráficas"), ("yaml", "configuración YAML")]:
    ok = importlib.util.find_spec(mod) is not None
    print(f"  {mod:<11} {'✓' if ok else '✗ falta'}  → {para}")

# El resto de la guía usa humlicek si numba está instalado, si no wofz (exacto, más lento)
VOIGT = "humlicek" if importlib.util.find_spec("numba") else "wofz"
print("backend de Voigt para esta guía:", VOIGT)

In [ ]:
# Estilo de las gráficas de la guía
import matplotlib.pyplot as plt
import numpy as np

AZUL, NARANJA, INK, INK_2, MUTED, GRID, AXIS, SURFACE = (
    "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb")
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": AXIS,
    "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
    "legend.frameon": False, "legend.labelcolor": INK_2,
})

## 2. Las tres piezas del forward

Todo cálculo junta tres cosas:

| Pieza | Qué es | Cómo se crea |
|---|---|---|
| **Líneas** (`SpectralData`) | Parámetros de las líneas de HITRAN + funciones de partición | `from_hapi("NH3", nu_min, nu_max, iso_id="all")` (descarga o caché) o `from_files(...)` (archivos `.par` propios) |
| **Camino** | Por dónde viaja la luz: presión, temperatura y concentración de cada gas | `HomogeneousPath(...)` para standoff (horizontal, homogéneo); `AtmosphericProfile(...)` o `us_standard_atmosphere(...)` para una columna vertical |
| **Cálculo** | La transmitancia monocromática | `compute_transmittance(...)` (Python directo), `run_forward(sd, camino, yaml)` o `run_path(yaml)` (todo en el YAML) |

Para un caso standoff típico basta con `run_path` (sección 5). Las secciones 3 y 4 muestran las piezas por separado.

## 3. Forward rápido: un gas en un camino standoff

Amoniaco a **1 ppm en 100 m**, a 1013.25 hPa y 296 K, en 900–1000 cm⁻¹ (10–11.1 µm).

In [ ]:
from pylblrtm import HomogeneousPath, compute_transmittance, from_hapi

# 1) Líneas: todos los isotopólogos (la intensidad de HITRAN ya incluye su abundancia)
sd = from_hapi("NH3", nu_min=900, nu_max=1000, iso_id="all")
print(sd)

# 2) Camino standoff: longitud [m] y concentración de cada gas como fracción molar (1 ppm = 1e-6)
path = HomogeneousPath(path_length_m=100.0, vmr={"NH3": 1e-6},
                       pressure_hpa=1013.25, temperature_k=296.0)
print(path)

# 3) Cálculo
nu, T = compute_transmittance(sd, path, nu_min_cm1=900, nu_max_cm1=1000,
                              step_cm1=0.01, voigt=VOIGT, n_jobs=-1)
print(f"{nu.size} puntos · T mínima {T.min():.3f} en {nu[T.argmin()]:.2f} cm⁻¹")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(1e4 / nu, T, color=AZUL, linewidth=1.0)
ax.set_xlabel("Longitud de onda [µm]")
ax.set_ylabel("Transmitancia")
ax.set_title("NH3 · 1 ppm × 100 m · 1013.25 hPa, 296 K", color=INK, loc="left")
ax.set_ylim(min(0.0, T.min()) - 0.02, 1.02)
fig.tight_layout()
plt.show()

**Parámetros de `compute_transmittance`**

| Parámetro | Default | Qué controla |
|---|---|---|
| `spectral_data` | — | Las líneas (`from_hapi` / `from_files`) |
| `profile` | — | El camino (`HomogeneousPath`, `AtmosphericProfile`) |
| `nu_min_cm1`, `nu_max_cm1` | — | Rango espectral [cm⁻¹]. µm → cm⁻¹: `1e4 / µm` |
| `step_cm1` | `0.001` | Paso de la rejilla [cm⁻¹]. 0.01 alcanza a 1 atm; 0.001 para presiones bajas |
| `line_cutoff_cm1` | `25.0` | Hasta dónde llega cada línea desde su centro [cm⁻¹] |
| `mu_view` | `1.0` | Coseno del ángulo cenital (solo perfiles verticales) |
| `voigt` | `"wofz"` | `"wofz"` exacto · `"humlicek"` rápido (numba), < 1e-4 de diferencia |
| `n_jobs` | `1` | Hilos; `-1` = todos los núcleos. No cambia el resultado |
| `wing_tolerance` | `None` | Con `humlicek`: alas lejanas en rejillas gruesas, p.ej. `1e-4` → varias veces más rápido |
| `continuum` | `False` | Continuo MT_CKD de H2O y CO2 |

Devuelve `(nu, T)`: la rejilla en cm⁻¹ y la transmitancia monocromática.

## 4. Varios gases a la vez

Se cargan las líneas de cada gas, se juntan con `SpectralData.combine` y el camino lleva la concentración de cada uno. Cada gas que tenga líneas debe estar en `vmr` (si no, sus líneas se ignoran con un aviso).

In [ ]:
from pylblrtm import SpectralData

gases = {"H2O": 0.0188, "CO2": 420e-6, "NH3": 1e-6}        # fracción molar
sd_mix = SpectralData.combine([from_hapi(g, 900, 1000, iso_id="all") for g in gases])
path_mix = HomogeneousPath(100.0, gases, pressure_hpa=1013.25, temperature_k=296.0)

nu, T_lineas = compute_transmittance(sd_mix, path_mix, 900, 1000, step_cm1=0.01, voigt=VOIGT, n_jobs=-1)
_,  T_cont   = compute_transmittance(sd_mix, path_mix, 900, 1000, step_cm1=0.01, voigt=VOIGT, n_jobs=-1,
                                     continuum=True)
print(sd_mix)
print(f"T media sin continuo {T_lineas.mean():.4f} · con continuo {T_cont.mean():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(1e4 / nu, T_lineas, color=AZUL, linewidth=1.0, label="solo líneas")
ax.plot(1e4 / nu, T_cont, color=NARANJA, linewidth=1.0, label="líneas + continuo")
ax.set_xlabel("Longitud de onda [µm]")
ax.set_ylabel("Transmitancia")
ax.set_title("H2O 1.88 % + CO2 420 ppm + NH3 1 ppm · 100 m", color=INK, loc="left")
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

## 5. Todo desde un YAML con `run_path`

La forma recomendada para casos standoff: el escenario completo (gases, concentraciones, camino, rango, salidas) va en un YAML y `run_path` descarga las líneas, arma el camino y calcula. El ejemplo completo del repo es [`examples/standoff_7gases.yaml`](../examples/standoff_7gases.yaml) (7 gases, 600–2500 cm⁻¹). Aquí uno corto, escrito como dict de Python (en un archivo `.yaml` es el mismo contenido):

```yaml
spectral:
  nu_min_cm1: 900.0
  nu_max_cm1: 1100.0
  step_cm1: 0.01
calculation:
  voigt: humlicek          # o wofz
  n_jobs: -1
  continuum: true
path:
  length_m: 100.0
  pressure_hpa: 1013.25
  temperature_k: 296.0
  isotopologues: all       # all | 1 | [1, 2]
  gas_units: ppm           # vmr (fracción molar) | ppm | atm (presión parcial)
  gases: {H2O: 18800, CO2: 420, NH3: 1}
output:
  per_gas: true            # transmitancia de cada gas
  bin_um: 0.02             # promedio en bines de 0.02 µm
  attenuation_db_m: true   # -10/L · log10(T) [dB/m]
  save: {enabled: false, path: "OUT/", format: csv, prefix: "mi_caso"}
```

In [ ]:
from pylblrtm import run_path

config = {
    "spectral": {"nu_min_cm1": 900.0, "nu_max_cm1": 1100.0, "step_cm1": 0.01},
    "calculation": {"voigt": VOIGT, "n_jobs": -1, "continuum": True},
    "path": {"length_m": 100.0, "pressure_hpa": 1013.25, "temperature_k": 296.0,
             "isotopologues": "all", "gas_units": "ppm",
             "gases": {"H2O": 18800, "CO2": 420, "NH3": 1}},
    "output": {"per_gas": True, "bin_um": 0.02, "attenuation_db_m": True},
}
res = run_path(config)               # o run_path("ruta/a/mi_caso.yaml")
print(res)

**Qué trae el resultado (`ForwardResult`)**

| Campo | Forma | Qué es |
|---|---|---|
| `wavenumber_cm1` | (n_nu,) | Rejilla en cm⁻¹ |
| `transmittance` | (n_nu,) | Transmitancia total monocromática |
| `transmittance_per_gas` | {gas: (n_nu,)} | Cada gas solo (`per_gas`); su producto es el total |
| `wavelength_um` | (n_bins,) | Centros de los bines [µm], crecientes (`bin_um`) |
| `transmittance_binned` | (n_bins,) | Total promediado en cada bin |
| `transmittance_per_gas_binned` | {gas: (n_bins,)} | Cada gas promediado en cada bin |
| `attenuation_db_m` | (n_bins,) | Atenuación total [dB/m] (`attenuation_db_m`) |
| `attenuation_per_gas_db_m` | {gas: (n_bins,)} | Atenuación de cada gas [dB/m] |
| `skipped_gases` | {gas: motivo} | Gases sin líneas en el rango (no absorben ahí) |

`res.save(path="OUT/", fmt="csv")` escribe `OUT/forward.csv` (rejilla) y `OUT/forward_binned.csv` (bines).

In [ ]:
lam = res.wavelength_um
A = res.attenuation_per_gas_db_m
envolvente = sum(A.values())          # suma de los gases

for gas, a in A.items():
    print(f"{gas:<5} A máx {a.max():.2e} dB/m en {lam[a.argmax()]:.2f} µm · media {a.mean():.2e} dB/m")
print(f"envolvente: máx {envolvente.max():.2e} dB/m en {lam[envolvente.argmax()]:.2f} µm")

# Un panel por serie (misma escala): ningún gas tapa a otro
paneles = list(A.items()) + [("Envolvente", envolvente)]
fig, axes = plt.subplots(len(paneles), 1, figsize=(10, 1.7 * len(paneles)), sharex=True, sharey=True)
for ax, (nombre, a) in zip(axes, paneles):
    ax.semilogy(lam, np.clip(a, 1e-8, None), color=INK if nombre == "Envolvente" else AZUL, linewidth=1.2)
    ax.text(0.005, 0.8, nombre, transform=ax.transAxes, color=INK, fontsize=10, fontweight="bold")
axes[-1].set_xlabel("Longitud de onda [µm]")
fig.supylabel("Atenuación [dB/m]", color=INK_2)
fig.tight_layout()
plt.show()

## 6. Atmósfera vertical

Para una columna vertical se usa un perfil por niveles en vez de un camino homogéneo:

- `us_standard_atmosphere(gases=[...])`: Atmósfera Estándar de EE.UU. 1976 con perfiles AFGL para H2O, CO2, O3, N2O, CH4 y CO (`list_profile_gases()`), de la superficie a 50 km.
- `AtmosphericProfile(pressure_hpa, temperature_k, vmr)`: un perfil propio, niveles ordenados del tope (presión menor) a la superficie, con un valor de VMR por nivel.

`mu_view` es el coseno del ángulo cenital: 1 = nadir; 0.5 = 60°, que duplica el camino.

In [ ]:
from pylblrtm import AtmosphericProfile, us_standard_atmosphere

sd_co2 = from_hapi("CO2", 700, 760, iso_id="all")
perfil = us_standard_atmosphere(gases=["CO2"])
print(perfil)

nu, T_nadir = compute_transmittance(sd_co2, perfil, 700, 760, step_cm1=0.01, voigt=VOIGT, n_jobs=-1)
_,  T_60    = compute_transmittance(sd_co2, perfil, 700, 760, step_cm1=0.01, voigt=VOIGT, n_jobs=-1,
                                    mu_view=0.5)

# Un perfil propio: 3 niveles, tope → superficie
mi_perfil = AtmosphericProfile(pressure_hpa=[100.0, 500.0, 1013.25],
                               temperature_k=[220.0, 255.0, 288.0],
                               vmr={"CO2": [420e-6] * 3})
_, T_propio = compute_transmittance(sd_co2, mi_perfil, 700, 760, step_cm1=0.01, voigt=VOIGT, n_jobs=-1)
print(f"T media · nadir {T_nadir.mean():.3f} · 60° {T_60.mean():.3f} · perfil propio {T_propio.mean():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(nu, T_nadir, color=AZUL, linewidth=0.8, label="nadir (mu_view=1)")
ax.plot(nu, T_60, color=NARANJA, linewidth=0.8, label="60° (mu_view=0.5)")
ax.set_xlabel("Número de onda [cm⁻¹]")
ax.set_ylabel("Transmitancia")
ax.set_title("CO2 · Atmósfera Estándar de EE.UU., columna completa", color=INK, loc="left")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

## 7. Parámetros del cálculo y cuándo cambiarlos

| Situación | Qué ajustar |
|---|---|
| Va lento | `voigt="humlicek"` (instalar `.[fast]`) y `n_jobs=-1`; además `wing_tolerance=1e-4` |
| Presiones bajas (estratosfera, cámaras de baja presión) | `step_cm1` menor (0.001): las líneas son más angostas |
| Comparar con un sensor de resolución fija | `bin_um` (promedio en bines de longitud de onda) |
| Ventana de 8–13 µm con agua | `continuum=True` (si no, la ventana sale más transparente de lo real) |
| Memoria | `tile_width_cm1` menor; no cambia el resultado |

`humlicek`, `n_jobs` y `tile_width_cm1` no cambian el resultado más allá de 1e-4 (verificado en los tests). `step_cm1` y `line_cutoff_cm1` sí son parte de la física del cálculo.

## 8. Errores y excepciones

Las descargas pueden fallar por razones que no son errores del código. Cada una tiene su excepción (todas importables desde `pylblrtm`):

| Excepción | Cuándo | Qué hacer |
|---|---|---|
| `HITRANNoLinesError` | La molécula no tiene líneas en ese rango (p.ej. CO en 8–14 µm) | Nada: no absorbe ahí. Con `iso_id="all"` se saltan los isotopólogos sin líneas; solo falla si ninguno tiene |
| `HITRANNotAvailableError` | HITRANonline no la sirve línea por línea (SF6, CF4, NF3, ClONO2) | Solo existen como secciones eficaces, que la librería aún no lee |
| `HITRANQuotaError` | Se superó el límite diario de consultas a HITRAN (por IP) | Esperar al día siguiente; lo ya descargado sigue funcionando desde la caché |
| `HITRANConnectionError` | Sin conexión a hitran.org y sin caché | Revisar internet, o pasar una caché con `cache_dir` |

`run_path` ya maneja las dos primeras: el gas se omite, queda en `res.skipped_gases` y su transmitancia es 1.

In [ ]:
from pylblrtm import HITRANNoLinesError, HITRANNotAvailableError, HITRANQuotaError

for gas in ["NH3", "CO", "SF6"]:
    try:
        d = from_hapi(gas, 900, 1000, iso_id="all")
        print(f"✓ {gas}: {d.n_lines} líneas")
    except HITRANNoLinesError as err:
        print(f"· {gas}: sin líneas en 900–1000 cm⁻¹ ({type(err).__name__})")
    except HITRANNotAvailableError as err:
        print(f"· {gas}: no disponible línea por línea ({type(err).__name__})")
    except HITRANQuotaError:
        print(f"! {gas}: límite diario de HITRAN; reintentar mañana")

## 9. Unidades, convenciones y limitaciones

**Unidades**

- Número de onda en cm⁻¹; longitud de onda en µm (`µm = 1e4 / cm⁻¹`). Presión en hPa (= mbar); temperatura en K.
- Concentración: fracción molar (VMR; 1 ppm = 1e-6). En el YAML, `gas_units: ppm` o `atm` (presión parcial: VMR = p_gas / P_total).
- Atenuación: `A = −10/L · log10(T)` en dB/m.

**Convenciones**

- Las intensidades de HITRAN ya incluyen la abundancia natural de cada isotopólogo: se da la concentración **total** de la molécula y se cargan todos sus isotopólogos (`iso_id="all"`). `get_abundance(molécula, iso)` devuelve la abundancia si hace falta trabajar con un isotopólogo específico.
- Las líneas vienen de HITRANonline (edición HITRAN2024) y las funciones de partición son TIPS-2025. Se descargan una vez y quedan en `~/.pylblrtm/hitran_cache/`.

**Limitaciones actuales**

- Continuo: solo H2O (620–2780 cm⁻¹) y CO2 (< 2000 cm⁻¹), MT_CKD. Sin absorción inducida por colisiones de O2/N2.
- Sin line mixing (afecta la forma de las ramas Q del CO2).
- Transmitancia monocromática: no aplica la función instrumental de un sensor (solo promedio en bines con `bin_um`).

**Un ejemplo completo en el repo:** [`examples/standoff_7gases.yaml`](../examples/standoff_7gases.yaml): 7 gases (H2O, CO2, O3, N2O, CO, CH4, O2), 600–2500 cm⁻¹, con continuo, atenuación por gas en bines de 0.02 µm. Se corre con `run_path("examples/standoff_7gases.yaml")` desde la raíz del repo y la envolvente es `sum(res.attenuation_per_gas_db_m.values())`. La referencia de la API está en el `README.md`.